# The Active Learning Loop

This notebook explains what active learning is and how the loop works.

## Problem

Labelling data is expensive and time-consuming. Say you have 10,000 audio clips but can only afford to label 100. Which 100 should you label to get the best model?

This is the core challenge that active learning solves:

> Given vast amounts of raw data and limited annotation resources, which data should be prioritised for labelling?

## The loop

1. Start with a small labelled set and a large unlabelled set
2. Train the model on the labelled set
3. Use the model to score unlabelled samples (e.g. by utility)
4. Select the most informative samples
5. Label them (or simulate labelling from existing ground truth)
6. Add to the labelled set
7. Repeat from **step 2**


## Labelled vs unlabelled indices

The pipeline tracks two sets of indices:
- `labeled_indices`: samples we can train on
- `unlabeled_indices`: samples we have not yet selected

In [1]:
import numpy as np

total_samples = 100

# Start with 5 labelled, 95 unlabelled
labeled_indices = [0, 1, 2, 3, 4]
unlabeled_indices = list(range(5, 100))

print(f"Labelled: {len(labeled_indices)} samples")
print(f"Unlabelled: {len(unlabeled_indices)} samples")

Labelled: 5 samples
Unlabelled: 95 samples


## Sampling strategies

How do we choose which unlabelled samples to label next?

### Random sampling (baseline)

Pick samples at random. Simple but not optimal.

In [2]:
def sample_random(unlabeled_indices, n_samples=5):
    selected = np.random.choice(unlabeled_indices, size=n_samples, replace=False)
    return selected.tolist()

selected = sample_random(unlabeled_indices, n_samples=5)
print(f"Randomly selected: {selected}")

Randomly selected: [32, 87, 71, 74, 72]


### Uncertainty sampling

Pick samples the model is most uncertain about. If the model predicts probabilities [0.51, 0.49], it is very uncertain and this sample would be informative. If it predicts [0.99, 0.01], it is confident and the sample is less useful.

In [3]:
import torch
import torch.nn.functional as F

def sample_uncertainty(model, embeddings, unlabeled_indices, n_samples=5):
    model.eval()
    with torch.no_grad():
        X_unlabeled = embeddings[unlabeled_indices]
        logits = model(X_unlabeled)
        probs = F.softmax(logits, dim=1)

        # Uncertainty = 1 - max probability
        max_probs, _ = torch.max(probs, dim=1)
        uncertainty = 1 - max_probs

        top_k_indices = torch.topk(uncertainty, k=n_samples).indices
        selected = [unlabeled_indices[i] for i in top_k_indices.tolist()]
    return selected

# Demo with fake data
fake_model = torch.nn.Linear(10, 5)
fake_embeddings = torch.randn(100, 10)

selected = sample_uncertainty(fake_model, fake_embeddings, unlabeled_indices, n_samples=5)
print(f"Uncertainty-based selection: {selected}")

Uncertainty-based selection: [24, 11, 85, 36, 61]


## Moving samples from unlabelled to labelled

After selecting samples, we move them from the unlabelled set to the labelled set.

In [4]:
def add_samples(labeled_indices, unlabeled_indices, selected_indices):
    for idx in selected_indices:
        if idx in unlabeled_indices:
            unlabeled_indices.remove(idx)
            labeled_indices.append(idx)
    return labeled_indices, unlabeled_indices

print(f"Before: Labelled={len(labeled_indices)}, Unlabelled={len(unlabeled_indices)}")
labeled_indices, unlabeled_indices = add_samples(labeled_indices, unlabeled_indices, selected)
print(f"After:  Labelled={len(labeled_indices)}, Unlabelled={len(unlabeled_indices)}")

Before: Labelled=5, Unlabelled=95
After:  Labelled=10, Unlabelled=90


## A complete cycle

Putting it together: train on the labelled set, select uncertain samples, add them, repeat.

In [5]:
import torch.nn as nn
import torch.optim as optim

n_samples = 100
n_features = 10
n_classes = 3

embeddings = torch.randn(n_samples, n_features)
labels = torch.randint(0, n_classes, (n_samples,))

model = nn.Linear(n_features, n_classes)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

labeled_indices = [0, 1, 2, 3, 4]
unlabeled_indices = list(range(5, n_samples))

print(f"Initial state: Labelled={len(labeled_indices)}, Unlabelled={len(unlabeled_indices)}")

# Train on labelled data
model.train()
for epoch in range(10):
    optimizer.zero_grad()
    logits = model(embeddings[labeled_indices])
    loss = criterion(logits, labels[labeled_indices])
    loss.backward()
    optimizer.step()
print(f"Training loss: {loss.item():.4f}")

# Select uncertain samples
selected = sample_uncertainty(model, embeddings, unlabeled_indices, n_samples=5)
print(f"Selected: {selected}")

# Add to labelled set
labeled_indices, unlabeled_indices = add_samples(labeled_indices, unlabeled_indices, selected)
print(f"After cycle: Labelled={len(labeled_indices)}, Unlabelled={len(unlabeled_indices)}")

Initial state: Labelled=5, Unlabelled=95
Training loss: 0.9723
Selected: [20, 50, 51, 64, 78]
After cycle: Labelled=10, Unlabelled=90


## Summary

Active learning in four steps:
1. **Train** on the current labelled data
2. **Score** unlabelled samples using a sampling strategy
3. **Select** the top-scoring samples
4. **Add** them to the labelled set and repeat

The benefit is better models with fewer labels. In BaseAL, the `ActiveLearner` class manages this loop for you. See notebook 05 for a full working example.